In [3]:

from hybrid_automaton import Automaton, State, Transition
from typing import Dict, Tuple
import numpy as np
import time

In [4]:
GRAVITY = -9.81
RESTITUTION = 0.8  # Energy loss coefficient

In [5]:
def flying_flow(x: Automaton.Runtime.ContinousState, aux_x: Dict[str, Automaton.Runtime.AuxiliaryState], 
                u: Dict[str, Automaton.Runtime.ControlInput], cfg: Dict, clk: Automaton.Runtime.Clock) -> np.array:
    """Free fall dynamics - x = [y, v]"""
    # dx/dt = [v, g]
    return np.array([x.get_continous_state()[0], GRAVITY])

def ground_flow(x: Automaton.Runtime.ContinousState, aux_x: Dict[str, Automaton.Runtime.AuxiliaryState], 
                u: Dict[str, Automaton.Runtime.ControlInput], cfg: Dict, clk: Automaton.Runtime.Clock) -> np.array:
    """On ground (stationary)"""
    return np.array([0.0, 0.0])

## Guards

In [6]:

def hits_ground(x: Automaton.Runtime.ContinousState, aux_x: Dict[str, Automaton.Runtime.AuxiliaryState], 
                u: Dict[str, Automaton.Runtime.ControlInput], cfg: Dict, clk: Automaton.Runtime.Clock) -> bool:
    """Ball hits the ground"""
    return x.get_continous_state()[0] <= 0.0 and x[1] < 0  # y <= 0 and v < 0

def bounces_up(x: Automaton.Runtime.ContinousState, aux_x: Dict[str, Automaton.Runtime.AuxiliaryState], 
                u: Dict[str, Automaton.Runtime.ControlInput], cfg: Dict, clk: Automaton.Runtime.Clock) -> bool:
    """Ball has velocity to bounce"""
    return abs(x.get_continous_state()[0]) > 0.1  # Threshold for bouncing

def stops_bouncing(x: Automaton.Runtime.ContinousState, aux_x: Dict[str, Automaton.Runtime.AuxiliaryState], 
                u: Dict[str, Automaton.Runtime.ControlInput], cfg: Dict, clk: Automaton.Runtime.Clock) -> bool:
    """Ball has too little energy to bounce"""
    return abs(x.get_continous_state()[0]) <= 0.1

## Resets


In [7]:
def bounce_reset(x: Automaton.Runtime.ContinousState, aux_x: Dict[str, Automaton.Runtime.AuxiliaryState], 
                u: Dict[str, Automaton.Runtime.ControlInput], cfg: Dict, clk: Automaton.Runtime.Clock
                ) -> Tuple[np.array, Dict[str, Automaton.Runtime.AuxiliaryState], Dict[str, Automaton.Runtime.ControlInput]]:
    """Reverse velocity with energy loss"""
    x.set_continous_state(np.array([0.0, -x[1] * RESTITUTION]))
    return x, aux_x, u

def stop_reset(x: Automaton.Runtime.ContinousState, aux_x: Dict[str, Automaton.Runtime.AuxiliaryState], 
                u: Dict[str, Automaton.Runtime.ControlInput], cfg: Dict, clk: Automaton.Runtime.Clock
                ) -> Tuple[np.array, Dict[str, Automaton.Runtime.AuxiliaryState], Dict[str, Automaton.Runtime.ControlInput]]:
    """Stop the ball"""
    x.set_continous_state(np.array([0.0, 0.0]))
    return x, aux_x, u

## INtegration function

## States Initialization

In [8]:
# States
flying = State(name="FLYING", initial=True, flow=flying_flow)
ground = State(name="GROUND", flow=ground_flow)

In [9]:
# Transitions
flying.add_transition(Transition("hit_ground", ground, guards=[hits_ground], 
                                reset=bounce_reset, priority=1))
ground.add_transition(Transition("bounce_up", flying, guards=[bounces_up], priority=1))
ground.add_transition(Transition("stop", ground, guards=[stops_bouncing], priority=2))

In [12]:
# Create automaton
ball = Automaton(
    name="Bouncing Ball", 
    states=[flying, ground], 
    integration_function=None, 
    on_entry=lambda: print('starting bouncing ball!'), 
    on_exit=lambda: print('ending bouncing ball'))

# Drop from 5 meters: x = [y, v]
x0 = np.array([5.0, 0.0])

import asyncio
await ball.activate(x0=x0, real_time_mode=False, dt=0.1)

activating automaton 'Bouncing Ball'...
automaton 'Bouncing Ball' evaluation loop worker starting.


AttributeError: 'numpy.ndarray' object has no attribute 'get_continous_state'

In [8]:
print("Dropping ball from 5m height...")
print()

# Simulate
bounce_count = 0
for i in range(500):
    result = ball.step()
    if i % 50 == 0:
        print(f"t={i*0.01:.2f}s | State: {ball.q.name:7s} | Height: {ball.x[0]:5.2f}m | Velocity: {ball.x[1]:6.2f}m/s")
    if result and result.transition_taken:
        if result.transition_taken.name == "hit_ground":
            bounce_count += 1
            print(f"  💥 BOUNCE #{bounce_count}")

print(f"\nTotal bounces: {bounce_count}")
print()

Dropping ball from 5m height...

t=0.00s | State: FLYING  | Height:  5.00m | Velocity:  -0.10m/s
t=0.50s | State: FLYING  | Height:  3.75m | Velocity:  -5.00m/s
t=1.00s | State: FLYING  | Height:  0.05m | Velocity:  -9.91m/s
  💥 BOUNCE #1
t=1.50s | State: FLYING  | Height:  2.74m | Velocity:   3.30m/s
t=2.00s | State: FLYING  | Height:  3.18m | Velocity:  -1.61m/s
t=2.50s | State: FLYING  | Height:  1.18m | Velocity:  -6.51m/s
  💥 BOUNCE #2
t=3.00s | State: FLYING  | Height:  1.61m | Velocity:   3.41m/s
t=3.50s | State: FLYING  | Height:  2.11m | Velocity:  -1.50m/s
t=4.00s | State: FLYING  | Height:  0.16m | Velocity:  -6.40m/s
  💥 BOUNCE #3
t=4.50s | State: FLYING  | Height:  1.45m | Velocity:   0.85m/s

Total bounces: 3

